# Figure 1: SLC-40 and BCHH deployment geometry

This notebook generates the two-panel location figure for the Falcon 9 / BCHH paper.

The workflow is deliberately metadata-driven:

- launch-pad coordinates come from KML;
- deployed sensor coordinates come from time-selected StationXML;
- UTM coordinates are calculated from the StationXML latitude/longitude values;
- reusable functions live in `figure1_utils.py`;
- project-specific paths, dates, channel mappings, and output names remain explicit here.

The basemap cells require internet access because `contextily` downloads map tiles.

## 1. Imports

Keep `figure1_utils.py` in the same directory as this notebook, or place it somewhere on your Python path.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from obspy import UTCDateTime
from pyproj import Geod, Transformer

from figure1_utils import (
    get_sensor_location,
    inventory_stations_to_dataframe,
    load_station_sensor_dataframe,
    make_figure1,
    read_kml_points,
    save_figure,
)

pd.set_option("display.max_columns", None)

## 2. Project configuration

Only edit this cell when moving the notebook or changing metadata files. Relative paths make the notebook portable and avoid embedding a user-specific absolute output path.

In [ ]:
# Input metadata
KML_FILE = Path("launchpads_cameras.kml")
STATIONXML_FILE = Path("../03_stationxml/KSC.xml")

# Output location
OUTDIR = Path("../figures")
FIGURE_STEM = "fig01_slc40_bchh_location"

# Metadata interval containing the 1 September 2016 deployment
STARTTIME = UTCDateTime("2016-09-01T00:00:00")
ENDTIME = UTCDateTime("2016-09-02T00:00:00")

# StationXML selection
NETWORK_CODE = "1R"
STATION_CODE = "BCHH"
LOCATION_CODE = "10"

# Coordinate reference systems
GEOGRAPHIC_CRS = "EPSG:4326"   # WGS84 longitude/latitude
UTM_CRS = "EPSG:32617"         # WGS84 / UTM zone 17N

# Map placemark names
SLC40_KML_NAME = "SLC40"
SLC41_KML_NAME = "SLC41"

# Map the six recorded channels onto four physical sensors.
CHANNEL_TO_SENSOR = {
    "DHZ": "Seismometer",
    "DHN": "Seismometer",
    "DHE": "Seismometer",
    "DD1": "HD1",
    "DD2": "HD2",
    "DD3": "HD3",
}

# Short labels used in Panel B.
CHANNEL_TO_LABEL = {
    "DHZ": "BCHH",
    "DHN": "BCHH",
    "DHE": "BCHH",
    "DD1": "HD1",
    "DD2": "HD2",
    "DD3": "HD3",
}

## 3. Read launch-pad coordinates from KML

The KML is treated as the authoritative source for fixed infrastructure and camera locations. The full table is retained for inspection, while `SLC40` and `SLC41` are selected by placemark name.

In [ ]:
kml_points = read_kml_points(KML_FILE)

locations_df = (
    pd.DataFrame.from_dict(kml_points, orient="index")
    .rename_axis("kml_id")
    .reset_index()
)

SLC40 = kml_points[SLC40_KML_NAME]
SLC41 = kml_points[SLC41_KML_NAME]

display(locations_df)
print("SLC-40:", SLC40)
print("SLC-41:", SLC41)

## 4. Read and subset StationXML

The inventory is restricted to metadata epochs overlapping 1–2 September 2016. Channel coordinates are preferred over station-level coordinates because the three infrasound sensors occupied separate positions.

In [ ]:
inventory_event, channels_df, BCHH_SENSORS = (
    load_station_sensor_dataframe(
        stationxml_file=STATIONXML_FILE,
        starttime=STARTTIME,
        endtime=ENDTIME,
        network_code=NETWORK_CODE,
        station_code=STATION_CODE,
        location_code=LOCATION_CODE,
        channel_to_sensor=CHANNEL_TO_SENSOR,
        channel_to_label=CHANNEL_TO_LABEL,
        source_crs=GEOGRAPHIC_CRS,
        target_crs=UTM_CRS,
    )
)

stations_df = inventory_stations_to_dataframe(inventory_event)

print(inventory_event)
display(stations_df)
display(channels_df)
display(BCHH_SENSORS)

## 5. Derive the BCHH reference point and coordinate tools

Panel A needs one representative BCHH point. It is derived from the physical seismometer row rather than entered separately.

The geodesic and projection objects are passed explicitly into the plotting function; they are not hidden module globals.

In [ ]:
BCHH = get_sensor_location(
    sensors_df=BCHH_SENSORS,
    sensor_name="Seismometer",
    display_name="BCHH",
)

GEOD = Geod(ellps="WGS84")
LL_TO_UTM = Transformer.from_crs(
    GEOGRAPHIC_CRS,
    UTM_CRS,
    always_xy=True,
)
UTM_TO_LL = Transformer.from_crs(
    UTM_CRS,
    GEOGRAPHIC_CRS,
    always_xy=True,
)

print("BCHH reference point:", BCHH)

## 6. Generate Figure 1

`make_figure1` receives every data object and coordinate transformer explicitly. This makes dependencies clear and prevents the figure code from silently relying on notebook globals.

In [ ]:
fig, axes = make_figure1(
    slc40=SLC40,
    slc41=SLC41,
    bchh=BCHH,
    bchh_sensors=BCHH_SENSORS,
    geod=GEOD,
    ll_to_utm=LL_TO_UTM,
    utm_to_ll=UTM_TO_LL,
)

plt.show()

## 7. Save publication outputs

The figure is written to both PNG and PDF. Change `OUTDIR` or `FIGURE_STEM` in the configuration cell rather than editing this cell.

In [ ]:
output_paths = save_figure(
    fig=fig,
    output_directory=OUTDIR,
    filename_stem=FIGURE_STEM,
    extensions=("png", "pdf"),
    dpi=300,
)

for path in output_paths:
    print(path.resolve())

## Notes on reproducibility

- `KML_FILE` is the source of fixed launch-pad and camera coordinates.
- `STATIONXML_FILE` is the source of deployment-specific sensor positions.
- The time interval determines which StationXML epochs are selected.
- Easting and northing are derived from latitude and longitude using WGS84 / UTM zone 17N.
- The notebook contains no hardcoded BCHH sensor coordinates.
- Basemap appearance can change if the online tile provider updates its imagery or style.